# 🏊 第五课 · 矿池

## 本课目标

- 理解为什么个人矿工几乎不存在了
- 理解 Stratum 协议如何让矿工和矿池协作
- 理解 PPS 和 PPLNS 分红模式的区别
- 亲手模拟一个矿池的运行

> 这一课偏工程和经济学，代码比前几课少。

## 1. 问题：个人挖矿是买彩票

假设你有一台 Antminer S21（200 TH/s），在 600 EH/s 全网算力中：

```
你的份额 = 200 TH/s / 600,000,000 TH/s = 0.000000033%
```

按每天 144 个区块算，**你平均 21 年才能挖到一个块**。

但运气是不均匀的——你可能 40 年挖不到，也可能一年挖 3 个。

问题是：电费每天要交，收益完全不确定。这就是"收益方差"问题。

**矿池的解法**：1000 个你这样的矿工一起挖，挖到按贡献分。
每个人从"N 年中一次彩票"变成"每天稳定一点点收入"。

## 2. Stratum 协议

Stratum 是矿池和矿工之间通信的标准协议（JSON-RPC over TCP）。

工作流程：

```
矿工                    矿池服务器
  |                         |
  |-- mining.subscribe ---->|"我要加入，给我任务"
  |<-- 订阅确认 ------------|
  |                         |
  |<-- mining.notify -------|"新区块模板来了！"
  |   (job_id, prev_hash,   |
  |    coinbase1, coinbase2,|
  |    merkle_branches,     |
  |    version, nbits, ntime)|
  |                         |
  |  [矿工在自己的 nonce    |
  |   范围内暴力试...]      |
  |                         |
  |-- mining.submit ------->|"找到了一个 share！"
  |   (job_id, nonce, hash) |
  |                         |
  |<-- result: accept -------|"验证通过"
  |<-- (或) new job ---------|"新任务——可能因新区块"
```

**关键设计**：矿池给矿工的 share 难度远低于网络难度。
这样矿工能频繁提交 share（比如每分钟一个），矿池确认他在干活。

In [ ]:
import hashlib, time, random
from dataclasses import dataclass, field
from typing import List, Dict
from collections import defaultdict

@dataclass
class Share:
    # 矿工提交的一个"有效解"
    miner_name: str
    nonce: int
    hash: str
    difficulty: float  # 这个 share 满足的难度

class MiningPool:
    # 模拟一个矿池服务器
    def __init__(self, name: str, block_reward: float = 3.125):
        self.name = name
        self.block_reward = block_reward
        self.shares: List[Share] = []  # 所有提交的 share
        self.stats: Dict[str, dict] = defaultdict(
            lambda: {"shares": 0, "earned": 0.0}
        )
        self.total_blocks = 0

    def submit_share(self, miner_name: str, share: Share) -> bool:
        # 矿工提交了一个 share
        # 返回 True 如果这个 share 同时也是完整区块解
        h_int = int(share.hash, 16)
        share_target = (2**256) / share.difficulty

        # 验证 share 是否满足 share 难度
        if h_int >= share_target:
            return False  # 无效

        self.shares.append(share)
        self.stats[miner_name]["shares"] += 1

        # 检查是否满足网络难度（挖到真正的区块）
        network_target = (2**256) / 70_000_000_000_000
        if h_int < network_target:
            self.total_blocks += 1
            self._distribute_reward()
            return True  # 挖到区块了！
        return False

    def _distribute_reward(self):
        # PPLNS: 按最近 100 个 share 平均分配
        n = min(100, len(self.shares))
        recent = self.shares[-n:]
        for share in recent:
            self.stats[share.miner_name]["earned"] += \
                self.block_reward / n

class PoolMiner:
    # 连接矿池的矿工
    def __init__(self, name: str, pool: MiningPool, hashrate: int):
        self.name = name
        self.pool = pool
        self.hashrate = hashrate
        self.total_hashes = 0

    def mine_step(self, duration: float = 0.1):
        # 模拟一小段时间的挖矿
        hashes = int(self.hashrate * duration)
        share_diff = 1000  # share 难度（远低于网络难度）
        share_target = int((2**256) / share_diff)

        for _ in range(hashes):
            nonce = random.randint(0, 2**32 - 1)
            data = f"block|{self.name}|{nonce}|{time.time()}".encode()
            h = hashlib.sha256(hashlib.sha256(data).digest()).hexdigest()
            self.total_hashes += 1

            h_int = int(h, 16)
            if h_int < share_target:
                share = Share(self.name, nonce, h, share_diff)
                found = self.pool.submit_share(self.name, share)
                if found:
                    print(f"   🎉 {self.name} 挖到了完整的区块！")
                break

print("✅ 矿池模型已就绪")  

## 3. 运行矿池模拟

下面创建一个小矿池，加入几个不同算力的矿工，观察收益分配。

In [ ]:
# 创建矿池
pool = MiningPool("演示矿池", block_reward=3.125)

# 加入矿工（不同算力）
miners = [
    PoolMiner("你的 MacBook",       pool, hashrate=10_000),
    PoolMiner("一台游戏本",         pool, hashrate=50_000),
    PoolMiner("RTX 4090 工作站",   pool, hashrate=3_000_000_000),
    PoolMiner("Antminer S21 #1",   pool, hashrate=200_000_000_000_000),
    PoolMiner("Antminer S21 #2",   pool, hashrate=200_000_000_000_000),
    PoolMiner("Antminer S21 #3",   pool, hashrate=200_000_000_000_000),
]

total_hr = sum(m.hashrate for m in miners)
print(f"矿工数: {len(miners)}")
print(f"总算力: {total_hr:,.0f} H/s ({total_hr/1e12:.1f} TH/s)")
print()

# 模拟 30 轮
for round_num in range(30):
    for miner in miners:
        miner.mine_step(duration=0.03)

# 统计结果
print("\n📊 矿池统计：\n")
total_shares = sum(s["shares"] for s in pool.stats.values())
print(f"{'矿工':<18} | {'Shares':>7} | {'占比':>7} | {'收益 (BTC)':>12}")
print("-" * 52)
for name in sorted(pool.stats, key=lambda n: pool.stats[n]["shares"], reverse=True):
    s = pool.stats[name]
    pct = s["shares"] / total_shares * 100 if total_shares else 0
    print(f"{name:<18} | {s['shares']:>7} | {pct:>6.1f}% | {s['earned']:>12.8f}")

print()
print("💡 Share 数严格正比于算力——谁干得多谁拿得多")
print("💡 你的 MacBook 在矿池里照样赚不到钱——份额太小")  

## 4. 分红模式对比

矿池用不同方式分配收益：

| 模式 | 全称 | 原理 | 风险 | 适合 |
|------|------|------|------|------|
| **PPS** | Pay Per Share | 每个 share 固定收入 | 矿池承担运气风险 | 求稳的小矿工 |
| **PPLNS** | Pay Per Last N Shares | 挖到块时按最近 N 个 share 分 | 矿工共担 | 长期忠实矿工 |
| **FPPS** | Full PPS | PPS + 手续费分成 | 矿池承担 | 目前最主流 |
| **SOLO** | 单干 | 挖到全拿，挖不到归零 | 矿工全担 | 信仰充值 |

矿池收取 1-3% 的手续费作为运营成本。

## 5. 矿池的争议

矿池解决了收益方差问题，但也带来了**中心化风险**：

- 前 3 大矿池控制了 50%+ 的算力
- 如果它们串通，理论上可以发动 51% 攻击
- 2021 年中国禁令后算力向美国矿池集中

这是一个没有完美答案的权衡：
- 不要矿池 → 只有大资本能挖矿
- 要矿池 → 散户能参与，但算力集中

比特币社区通过 **Stratum V2** 协议试图缓解：让矿工自己构建区块模板，
而不是被动接受矿池的模板。这样矿池更接近"协调者"而非"控制者"。

## 本课小结

| 要点 | 记住 |
|------|------|
| 矿池存在的理由 | 平滑收益方差 |
| Stratum 协议 | JSON-RPC over TCP，矿工订阅+提交 |
| Share | 低难度解，证明矿工在工作 |
| PPLNS | 按最近 share 分配，公平但有波动 |
| 中心化问题 | 矿池集中算力，Stratum V2 在缓解 |

**下一课**：难度调整——比特币如何像一个恒温器一样自动维持 10 分钟出块。